In [ ]:
# Instalaivimas reikalingų paketų
#!pip install git+https://github.com/huggingface/transformers accelerate
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
model_name = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForVision2Seq.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_lt.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'Generated Caption', 'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        prompt = "USER: <image>\nASSISTANT: Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text. "
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
        output_ids = model.generate(**inputs, max_new_tokens=50)
        generated_caption = processor.decode(output_ids[0], skip_special_tokens=True)
        
    # Remove everything between "INST" and "leave"
        generated_caption = re.sub(r'USER:.*?image?', '', generated_caption, flags=re.DOTALL).strip()
        generated_caption = generated_caption.strip('?')

        # Extract the quoted caption
        match = re.search(r'"(.*?)"', generated_caption)
        if match:
            generated_caption = match.group(1)
        else:
            generated_caption = "No caption found."

        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_09.csv", sep=';', index=False)
print("Generavimas baigtas.")